# 03 · 数据进货：上传全蛋白组数据，自动验货、自动登记

## 这一章在整个项目里的位置

```
00 账号安全 ✅ → 01 建仓库 ✅ → 02 代码库工程化 → [03 数据进货] → 04–09 Databricks → 10–12 对外服务和 AI
```

> 先做完 02 号再做这一章：最后一步要把代码推到 GitHub，让 CI 检查。

## 背景：为什么需要这一章

你的 86 GB 全蛋白组预测数据，现在只在你的电脑上（和实验室的服务器上）。别人要用，得找你拷；三个月后，可能连你自己都说不清"当时用的是哪一份"。

公司里，数据的第一站是**云上的 raw 仓库**：原样存放、到了自动检查、登记在册。之后所有人、所有程序（比如 04 号的 Databricks）都从登记过的数据出发，而不是从某个人的电脑上拷。

**整章的比喻：仓库收货。**

```
送货（上传）──▶ 仓库门口 ──门铃──▶ 验货员（Lambda）──▶ 收货单（_registry/）
                datasets/                  │
                                           └──▶ 值班日志（CloudWatch Logs）
```

## 这一章做的 8 件事

| # | 做什么 | 用的技术 | 比喻 |
|---|---|---|---|
| 1 | 定规矩：每个文件放在哪 | S3 的键（key） | 货架编号 |
| 2 | 写一个自动验货员 | Lambda | 收货员 |
| 3 | 货一到就叫验货员 | S3 事件通知 | 门铃 |
| 4 | 给验货员发工牌 | IAM 角色 | 只能进收货区 |
| 5 | 用图纸把这些建出来 | Terraform（01 号学的） | 按图施工 |
| 6 | 先送两件小样试试 | 冒烟测试 | 正式进货前试一次 |
| 7 | 送 7.4 GB 的大货 | 分块上传 + 指纹核对 | 大件拆开运，到了核对 |
| 8 | 看验货员的值班日志 | CloudWatch Logs | 出了问题去哪查 |

**花费：约 0.2 美元/月**（7.4 GB 的存储）。Lambda 每月前 100 万次调用免费，日志很少。

> 不认识的词（Lambda、事件、IAM 角色），先看[基础知识](../../docs/primer_aws_databricks.md) §2.1、§2.4、§2.5。

---

## 准备

运行下面这一格：定义执行命令的小工具，连上 AWS，读出 raw 桶的名字。**不用看代码。**

In [ ]:
import json, os, subprocess
from pathlib import Path
import boto3
from qafi_platform import ingest
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ["AWS_PROFILE"] = "qafi"   # Terraform、aws 命令和 Python 都用 00 号登录的身份
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
s3 = boto3.Session(region_name="eu-central-1").client("s3")
RAW = json.loads(subprocess.run("terraform -chdir=infra/dev output -json bucket_names", shell=True, capture_output=True, text=True, check=True).stdout)["raw"]
run("aws sts get-caller-identity --query Arn --output text")
print("raw 桶:", RAW)

**结果怎么看**：`arn:aws:iam::186972156153:user/pipi-admin` 和 `raw 桶: qafi-186972156153-dev-raw`。

> 如果报错说登录过期：在终端运行 `aws login --profile qafi`，然后重新运行这一格。

---

## 第 1 步 · 定规矩：每个文件放在哪

### 在干什么
规定 raw 桶里文件的"地址格式"。

### 背景
仓库要有货架编号，货一多才找得到。数据也一样：三个月后，你要能一眼看出"这是哪份数据、哪个版本"。

### 用了什么技术，为什么
S3 里文件的地址叫**键**（key）。我们的规矩：

```
datasets/<数据集>/<版本>/<文件名>          数据本身：原样存放，永远不改
_registry/datasets/<...>/<文件名>.json     每个文件的收货单：由验货员写
```

例子：全蛋白组预测的压缩包放在 `datasets/qafi-predictions/v1.0/exhaustive_mutagenesis_data_v1.0.tar.gz`，它的收货单在 `_registry/datasets/qafi-predictions/v1.0/exhaustive_mutagenesis_data_v1.0.tar.gz.json`。

**名字要说清楚它是什么**：叫 `qafi-predictions` 而不是 `proteome`，看名字就知道这是 QAFI 的预测结果，不是公开数据，也不是训练数据。

**为什么"永远不改"**：raw 是原始记录（01 号讲过）。数据有了新版本，就放进新的版本文件夹（`v1.1/`），旧的留着。以后任何一个结果，都能追溯到"当时用的是哪一份数据"。

### 公司里什么情况怎么做
- **这种规矩写在团队的数据规范里**，不守规矩的文件会被验货员拒收（第 2 步）
- **数据源更新了**（比如 ClinVar 每月发新版）：按日期或版本号放进新文件夹，例如 `datasets/clinvar/2026-10/`

### 面试怎么说
> "Raw data lands under datasets/<dataset>/<version>/ and is never modified; new data means a new version prefix, so every downstream result traces back to its exact input."

---

## 第 2 步 · 自动验货员：Lambda

### 在干什么
写一个小程序：每个文件到了，就检查一遍，并写一张收货单。

### 背景
靠人检查：会忘、会漏，晚上和周末没人。坏数据一旦进来（传了一半、格式不对、名字乱写），后面所有的分析都会跟着错，而且很难查到源头。

### 用了什么技术，为什么
**Lambda**（[基础知识](../../docs/primer_aws_databricks.md) §2.4）：只上传一个函数，有事件才运行，跑完就停，按毫秒收费。没有文件来的时候，一分钱不花。

验货员检查三件事：

| 检查 | 防什么 |
|---|---|
| 地址符合第 1 步的规矩 | 文件乱放，以后找不到 |
| 文件不是空的 | 上传失败留下的空文件 |
| 文件开头的几个字节和它的类型对得上 | 名字叫 `.gz`，里面其实不是压缩文件 |

最后一项叫**文件签名**：每种文件格式开头都有固定的几个字节。比如 gzip 压缩文件永远以 `1f 8b` 开头。只看文件名不可靠，看开头的字节才可靠。

检查完写一张**收货单**（一个 JSON 文件），记下：结果、发现的问题、大小、指纹（第 7 步讲）、版本号、谁上传的、什么时候。

**验货员只看"箱子对不对"，不拆箱**：内容的检查（列对不对、有没有重复行）在 04 号用 Databricks 做，那里是处理大数据的地方。Lambda 适合几秒钟的小活，一次最长只能跑 15 分钟。

代码在 [`src/qafi_platform/raw_validator.py`](../../src/qafi_platform/raw_validator.py)，有 12 个单元测试，CI 每次都会跑。

### 公司里什么情况怎么做
- **Lambda 的两种打包方式**：只用到 boto3 的小函数，打成 zip 包（我们这样）；需要大依赖（pandas、模型）的，打成 Docker 镜像（10 号的 API 会这样）
- **Lambda 出错了**：AWS 会自动重试 2 次。还失败，公司里会把这个事件送进"死信队列"（dead-letter queue），等人来处理，不会丢
- **同一个事件可能送来两次**：所以验货员要能重复执行而不出错。我们的收货单直接覆盖，结果一样

### 面试怎么说
> "A small Lambda validates every raw file on arrival — naming convention, non-empty, file signature — and writes a JSON receipt; content-level checks happen later in Spark."

---

## 第 3 步 · 货一到就叫验货员：S3 事件

### 在干什么
设置好：`datasets/` 下面一来新文件，S3 就自动叫醒验货员。

### 背景
没有事件的话，就得有个程序每隔几分钟去看一眼"有没有新文件"（叫轮询）：既浪费，又有延迟。

### 用了什么技术，为什么
**S3 事件通知**（[基础知识](../../docs/primer_aws_databricks.md) §2.5），就像门铃。我们只让 `datasets/` 按门铃。

**收货单为什么放在另一个地方（`_registry/`）**：如果收货单也会按门铃，验货员写完一张单子，门铃又响，它又被叫醒、又写一张……无限循环，账单一直涨。这是事件驱动里最常见的事故，所以"**触发的地方**"和"**写入的地方**"一定要分开。

### 公司里什么情况怎么做
- **一个桶的事件要通知好几个程序**：用 EventBridge（AWS 的事件总线）来分发。我们只有一个验货员，直接通知就够了
- **怀疑门铃没响**（没有收货单）：先看第 8 步的日志里有没有这次运行的记录。没有记录，就是门铃的问题；有记录，就是验货员的问题

### 面试怎么说
> "S3 event notifications trigger the validator only for the datasets/ prefix; receipts go to a different prefix, so the function can never trigger itself."

---

## 第 4 步 · 给验货员发工牌

### 在干什么
给 Lambda 一个 IAM 角色，它只能做该做的事。

### 背景
01 号讲过**最小权限**：就算验货员被人利用了，损失也只限于这几件事。

### 用了什么技术，为什么
角色 `qafi-dev-raw-validator` 只有这些权限：

| 允许 | 范围 |
|---|---|
| 读文件 | 只能读 `datasets/` |
| 写收货单 | 只能写 `_registry/` |
| 用钥匙 | 只能用 01 号那把 KMS 钥匙（桶是加密的，读、写都要用钥匙） |
| 写日志 | 只能写自己的日志 |

它不能删文件，不能改 `datasets/` 里的数据，也碰不到别的桶。

另外还有一张反方向的许可：**允许 raw 桶叫醒这个 Lambda**，只给 raw 桶，别的桶叫不动它。

### 公司里什么情况怎么做
- **Lambda 报 AccessDenied**：看日志里是哪个操作被拒，在 Terraform 里加上那一项，走 Pull Request
- **最常见的坑**：桶用 KMS 加密，却忘了给钥匙的权限，结果读文件失败。我们的图纸里已经给了

### 面试怎么说
> "The function's role can read datasets/, write _registry/ and use the bucket's KMS key — nothing else."

---

## 第 5 步 · 用图纸建出来

### 在干什么
第 2–4 步的所有东西都写在图纸 [`infra/dev/ingest.tf`](../../infra/dev/ingest.tf) 里。用 01 号学的 init → plan → apply 建出来。

### 用了什么技术，为什么
这次新增 8 个资源：

| 资源 | 是什么 |
|---|---|
| `aws_iam_role` 等 3 个 | 验货员的工牌和权限（第 4 步） |
| `aws_cloudwatch_log_group` | 验货员的值班日志本，保留 30 天 |
| `aws_lambda_function` | 验货员本身 |
| `aws_lambda_permission` | 允许 raw 桶叫醒它 |
| `aws_s3_bucket_notification` | 门铃：`datasets/` 来了新文件就通知 |
| `aws_s3_bucket_lifecycle_configuration` | raw 桶的清理规则：上传中断留下的碎片，7 天后删掉（第 7 步讲） |

Terraform 会自动把 `raw_validator.py` 打成 zip 包传上去，所以图纸里多用了一个 archive 插件，init 时会下载它。

### 执行

In [ ]:
run("terraform -chdir=infra/dev init -input=false -no-color")

plan：只看不做。下面只列出要新建的资源和最后一行：

In [ ]:
run("set -o pipefail; terraform -chdir=infra/dev plan -input=false -no-color -out=dev.tfplan 2>&1 | grep -E 'will be|Plan:|No changes|Error'")

### 结果怎么看
- init：输出里出现 `hashicorp/archive`，就是新插件准备好了
- plan：8 行 `will be created`，最后一行 **`Plan: 8 to add, 0 to change, 0 to destroy.`**：只新建，不改、不删任何已有的东西

> 如果你看完整的 plan 输出，开头有一段 `Objects have changed outside of Terraform`，说 raw 桶的加密和版本控制"变了"。那不是别人改的：01 号的加密和版本控制是用单独的设置加上去的，而 Terraform 的记账本里，桶本身还记着它刚建好时的样子。apply 之后记账本更新，这段提示就消失了。

apply：真正建。大约一分钟：

In [ ]:
run("set -o pipefail; terraform -chdir=infra/dev apply -input=false -no-color dev.tfplan 2>&1 | grep -E 'complete|Error'")

### 结果怎么看
每个资源一行 `Creation complete`，最后一行 `Apply complete! Resources: 8 added, 0 changed, 0 destroyed.`

### 公司里什么情况怎么做
- **改了验货员的代码**：再跑一次 plan，会看到 `aws_lambda_function.raw_validator will be updated in-place`，因为代码包变了。apply 就把新代码部署上去了。用 Terraform 部署 Lambda，就是这样
- 其他和 01 号一样：改图纸 → Pull Request → CI 跑 plan → 同事审查 → apply

---

## 第 6 步 · 先送两件小样：冒烟测试

### 在干什么
正式传 7.4 GB 之前，先传两个小文件：一个好的；一个坏的，名字叫 `.gz`，里面却是普通文字。

### 背景
大文件要传几十分钟。先用几秒钟确认整条流程是通的：门铃响了，验货员醒了，权限够用，收货单写出来了。这叫**冒烟测试**（smoke test）：新机器先通电，看看冒不冒烟。

### 执行

In [ ]:
receipts = ingest.smoke_test(s3, RAW)

### 结果怎么看
两行：

```
datasets/smoke-test/v1/smoke_ok.tsv.gz     accepted
datasets/smoke-test/v1/smoke_bad.tsv.gz    rejected: content is not a .gz file (wrong file signature)
```

好的收下了；坏的被拒收，理由是"文件签名不对"。坏文件**没有被删**，它留在原地，收货单上写明了原因。公司里由人来决定怎么处理，通常是重新上传一份正确的。

> 刚 apply 完，门铃可能要一两分钟才生效。如果第一次运行等到超时，再运行一次这一格。

看一张收货单的全部内容：

In [ ]:
print(json.dumps(receipts[0], indent=2))

| 字段 | 意思 |
|---|---|
| `status` / `problems` | 收下还是拒收，为什么 |
| `size_bytes` | 文件大小 |
| `checksum` | 文件的指纹（第 7 步讲） |
| `version_id` | S3 给这个版本的编号。同名文件再传一次是新版本，旧的还在（01 号开的版本控制） |
| `uploaded_by` | 谁上传的：你的 AWS 用户的内部编号 |
| `uploaded_at` / `checked_at` | 什么时候到的，什么时候验的 |

这两个小样会一直留在 raw 桶的 `smoke-test` 下面，很小，不用管。

### 公司里什么情况怎么做
- **每次改了验货员、部署之后**，都先跑一遍冒烟测试，再处理真数据
- **等了两分钟还没有收货单**：去第 8 步看日志

### 面试怎么说
> "After every deploy I run a smoke test with one good and one deliberately broken file, so I know the whole path works before real data goes through it."

---

## 第 7 步 · 正式进货：7.4 GB 分块上传

### 在干什么
把 7.4 GB 的压缩包传到 `datasets/qafi-predictions/v1.0/`，并证明云上的文件和你电脑上的**一个字节都不差**。

**这份是什么，为什么也进 raw**：它是 QAFI 论文流程在实验室服务器上算出的全蛋白组预测，是模型的**输出**，不是训练用的输入。但新平台没法重新生成它（全蛋白组的特征不重算），所以对新平台来说，它就是一份从外面来的数据，同样要从大门进、有版本、有收货单。后面它有三个用处：
- 04 号：整理成表（它自己也不标准，比如 P16615 那 560 万行重复）
- 09 号：新模型对全蛋白组打分后，和它逐个对照，它是全蛋白组级别的参照答案
- 10 号：对外的预测 API 查的就是它整理后的结果

因为它不是从官网来的，上传时会写一句**来源说明**，收货单里就有，和公开数据记录官方地址是一回事。

### 背景
大文件一次性传，有两个问题：中途断一下，就得从头再来；S3 一次最多只收 5 GB。另外，传了 7.4 GB，怎么知道一个字节都没错？

### 用了什么技术，为什么
**分块上传**（multipart upload）：把文件切成 222 块，每块 32 MB，8 块同时传。哪块失败了只重传那一块，全部到齐后 S3 再拼回一个文件。比喻：大件家具拆开，分几辆车运，到了再组装。

**指纹核对**（checksum）：
1. 上传前，在你电脑上算一遍文件的指纹（算法叫 CRC64NVME，结果是一串字符）。文件只要差一个字节，指纹就完全不同
2. 每一块都带着自己的指纹，S3 收到后自己再算一遍，对不上就拒收
3. 传完后，把 S3 存的整个文件的指纹读回来，和你电脑上的比：一样，就证明一字不差

**碎片清理**：上传中途断了（关电脑、断网），已经传上去的块你看不见，但会收存储费。第 5 步加的规则会在 7 天后自动清掉它们。

**一个常见的坑**：S3 给每个文件一个 ETag，很多人以为它就是文件的 MD5 指纹。分块上传的文件，ETag 末尾会多一个块数（比如 `-222`），它**不是**文件的指纹，不能拿来核对。所以我们用 CRC64NVME。

### 执行
要多久取决于你家网络的上传速度：50 Mbps 大约 20 分钟，10 Mbps 大约 1.5 小时。传的时候插上电源，别合上电脑。如果断了，重新运行这一格就行：会从头传，旧碎片会被自动清理。

In [ ]:
TARBALL = ROOT.parent / "exhaustive_mutagenesis_data_v1.0.tar.gz"
SOURCE = "QAFI paper pipeline: proteome-wide predictions v1.0, built on the lab server"
stored = ingest.upload(s3, TARBALL, RAW, ingest.raw_key("qafi-predictions", "v1.0", TARBALL.name), source=SOURCE)

### 结果怎么看
- 第一行 `local CRC64NVME ...  (7.44 GB)`：你电脑上算出的指纹
- 中间每 5% 一行进度
- 最后一行 `S3 CRC64NVME ...  identical`：S3 上的指纹和本地的一样，一字不差

> **已经传过的不会再传**：raw 里已经有这个文件、指纹一样、收货单是收下的，会显示 `already in raw with the same checksum ... skipped`，几秒钟就结束，不会再传一遍 7.4 GB。这叫**幂等**：同样的操作重复运行，结果不变。全蛋白组压缩包在 2026-10-08 已经传好了，所以你现在运行这一格会看到 skipped，下面的收货单照样能看。

再看验货员给它开的收货单：

In [ ]:
print(json.dumps(ingest.wait_for_receipt(s3, RAW, stored["key"], stored["version_id"], timeout=120), indent=2))

### 结果怎么看
`"status": "accepted"`；`size_bytes` 是 7441605059；`checksum` 里是 `CRC64NVME`，值和上面的指纹一样；`source` 里是那句来源说明。

### 公司里什么情况怎么做
- **几十 GB 到几 TB**：用 AWS 命令行或 SDK 的分块上传就够了，就是我们这样
- **几百 TB，或者网络很慢**：用 AWS DataSync（专门的传输服务），甚至 Snowball（AWS 寄一个硬盘箱过来，装满寄回去）
- **医药公司**：指纹加上收货单（谁、什么时候、哪个版本、指纹是多少）就是**数据完整性**的证据。GxP 法规要求数据可追溯到人、时间和原始记录（ALCOA+ 原则），面试医药公司时可以提

### 面试怎么说
> "Large files go up as multipart uploads with a CRC64NVME full-object checksum: S3 verifies every part on arrival, and afterwards I compare the stored checksum with the local one. The ETag of a multipart object is not a content hash, so it can't be used for that."

---

## 第 8 步 · 看值班日志：CloudWatch Logs

### 在干什么
看验货员每次干活留下的记录。

### 背景
Lambda 在 AWS 上运行，你看不到它的屏幕。出了问题（等不到收货单、文件被拒收），第一件事就是看日志。

### 用了什么技术，为什么
**CloudWatch Logs** 是 AWS 收集日志的地方。验货员的日志自动写进 `/aws/lambda/qafi-dev-raw-validator`，保留 30 天。

我们用 **JSON 格式**：每一行都是一条结构化的记录（时间、级别、内容……），可以按字段搜索，比如"只看出错的"。

### 执行
最近 3 小时的全部日志：

In [ ]:
run("aws logs tail /aws/lambda/qafi-dev-raw-validator --since 3h --format short")

只看出错的：

In [ ]:
run("aws logs tail /aws/lambda/qafi-dev-raw-validator --since 3h --format short --filter-pattern '{ $.level = \"ERROR\" }'")

### 结果怎么看
第一格里，验货员每运行一次会留下几行记录，主要看这三种：
- `platform.start`：开始
- 程序自己写的一行：`"level": "INFO"` 加 `accepted ...`，或者 `"level": "ERROR"` 加 `rejected ...`
- `platform.report`：这次花了多少毫秒（`durationMs`）、用了多少内存（`maxMemoryUsedMB`）。Lambda 就按这个收费

第二格只剩一行：那个坏的小样。

### 公司里什么情况怎么做
- **没人会天天盯着日志**：公司里设置报警（CloudWatch Alarm），比如"验货员出错次数大于 0，就发邮件"。13 号讲监控和报警
- **排查一个具体文件**：按文件名搜日志，再对照它的收货单

### 面试怎么说
> "The function logs structured JSON to CloudWatch with 30-day retention, so I can filter by level or key; in production an alarm on the error metric replaces watching logs."

---

## 第 9 步 · 去网页上看一眼（可选）

| 去哪 | 看什么 |
|---|---|
| **S3** → `qafi-186972156153-dev-raw` | `datasets/` 和 `_registry/` 两个"文件夹"。点开一张收货单可以下载来看 |
| **Lambda** → `qafi-dev-raw-validator` → **Monitor** 标签 | 调用次数、出错次数、每次耗时的图 |
| **CloudWatch** → Log groups → `/aws/lambda/qafi-dev-raw-validator` | 和第 8 步一样的日志 |

---

## 提交代码

和 02 号一样：提交，推到 GitHub。CI 会自动跑新加的测试（验货员和上传工具一共 16 个）和图纸检查：

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 03 raw ingestion with an S3-triggered validator") && git push -q')

推送后打开 <https://github.com/pipi0616/qafi-ml-platform/actions>，5 个任务应该全绿。

> 如果你是在这一章的代码写好之后才做的 02 号，那时的第一次提交已经包含了这些文件。这里就没有新东西要提交，只推送一下。

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 定规矩 | S3 的键 | 数据可追溯到版本 |
| 2 验货员 | Lambda | 每个文件到了都自动检查，不靠人 |
| 3 门铃 | S3 事件通知 | 事件驱动，不用轮询；触发和写入分开，防止死循环 |
| 4 工牌 | IAM 角色 | 最小权限 |
| 5 建出来 | Terraform | 和其他资源一样，能审查、能重建 |
| 6 小样 | 冒烟测试 | 正式进货前确认流程通 |
| 7 大货 | 分块上传 + 指纹 | 大文件传得稳，而且一字不差 |
| 8 日志 | CloudWatch Logs | 出了问题知道去哪查 |

**一句话**：从现在起，任何文件进 raw 仓库，都会被自动检查、登记，并且能证明和原件一模一样。

## 面试题

1. **How do you get a large file into S3 reliably?**
   > "Multipart upload: parts go up in parallel and only failed parts are retried. Each part carries a checksum that S3 verifies, the full-object CRC64NVME is compared with the local one afterwards, and a lifecycle rule aborts incomplete uploads after a week so orphaned parts don't cost money."

2. **Why an S3 event plus Lambda rather than a scheduled job?**
   > "It's event-driven: work starts the moment a file arrives, there's no polling, it costs nothing when idle, and it scales with the number of files automatically."

3. **How do you stop a Lambda from triggering itself?**
   > "Never write to the location that triggers it. The trigger is filtered to datasets/ and the function writes receipts to _registry/."

4. **How do you prove data integrity end to end?**
   > "Checksums at upload, an immutable versioned raw zone, and a receipt per file recording who uploaded which version when, with its checksum. In GxP terms that covers attributable, original and accurate from ALCOA+."

5. **What happens if the Lambda fails?**
   > "S3 invokes it asynchronously, so Lambda retries twice; in production a dead-letter queue or on-failure destination then keeps the event so nothing is lost. The handler is idempotent, so a retry or a duplicate event gives the same receipt. An alarm on the error metric tells someone to look."

**下一章 04 · Databricks**：把这份数据放进 Databricks，变成可以用 SQL 查询的表，并做第一轮清洗。